In [1]:
import os
os.environ["GOOGLE_API_KEY"]=""

## Install libraries

In [2]:
!pip install -q youtube-transcript-api langchain-community langchain-google-genai \
               faiss-cpu tiktoken python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 13.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 49.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 74.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [8]:
!pip install -U langchain langchain-community langchain-core langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 5.3 MB/s eta 0:00:00
  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.13
    Uninstalling langchain-1.3.13:
      Successfully uninstalled langchain-1.3.13


In [7]:
!pip install langchain.text_splitter

ERROR: Could not find a version that satisfies the requirement langchain.text_splitter (from versions: none)
ERROR: No matching distribution found for langchain.text_splitter


In [9]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

/tmp/ipykernel_3315/4146846605.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


## Step 1a - Indexing (Document Ingestion)

In [19]:
from youtube_transcript_api import YouTubeTranscriptApi

ytt_api = YouTubeTranscriptApi()

transcript = ytt_api.fetch("LPZh9BOjkQs")

text = " ".join([item.text for item in transcript])

print(text)



[Submit subtitle corrections at criblate.com]
Imagine you happen across a short movie script that describes a scene between a person and their AI assistant. The script has what the person asks the AI, but the AI's response has been torn off. Suppose you also have this powerful magical machine that can take any text and provide a sensible prediction of what word comes next. You could then finish the script by feeding in what you have to the machine, seeing what it would predict to start the AI's answer, and then repeating this over and over with a growing script completing the dialogue. When you interact with a chatbot, this is exactly what's happening. A large language model is a sophisticated mathematical function that predicts what word comes next for any piece of text. Instead of predicting one word with certainty, though, what it does is assign a probability to all possible next words. To build a chatbot, what you do is lay out some text that describes an interaction between a user

In [20]:
transcript

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='[Submit subtitle corrections at criblate.com]\nImagine you happen across a short movie script that', start=1.14, duration=2.836), FetchedTranscriptSnippet(text='describes a scene between a person and their AI assistant.', start=3.976, duration=3.164), FetchedTranscriptSnippet(text="The script has what the person asks the AI, but the AI's response has been torn off.", start=7.48, duration=5.58), FetchedTranscriptSnippet(text='Suppose you also have this powerful magical machine that can take', start=13.06, duration=3.92), FetchedTranscriptSnippet(text='any text and provide a sensible prediction of what word comes next.', start=16.98, duration=3.98), FetchedTranscriptSnippet(text='You could then finish the script by feeding in what you have to the machine,', start=21.5, duration=4.006), FetchedTranscriptSnippet(text="seeing what it would predict to start the AI's answer,", start=25.506, duration=2.862), FetchedTranscriptSnippet(te

## Step 1b - Indexing (Text Splitting)

In [22]:
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_text_splitters import RecursiveCharacterTextSplitter

video_id = "LPZh9BOjkQs"

ytt_api = YouTubeTranscriptApi()

# Fetch transcript
transcript = ytt_api.fetch(video_id)

# Convert transcript to plain text
text = " ".join(item.text for item in transcript)

# Split into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.create_documents([text])

# Print first chunk
print(chunks[0].page_content)

# Number of chunks
print(f"Total Chunks: {len(chunks)}")

[Submit subtitle corrections at criblate.com]
Total Chunks: 11


In [23]:
len(chunks)

11

In [24]:
chunks[100]

IndexError: list index out of range

## Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [27]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2-preview"
)

vector_store = FAISS.from_documents(chunks, embeddings)

In [30]:
vector_store.index_to_docstore_id

{0: '15878bd1-a369-4e9b-a6f2-101be25300d1',
 1: '41361957-a47d-486d-84d1-0f68de99b8ee',
 2: 'f50f0d7c-e7e6-4ee2-bce7-57e92486dda9',
 3: 'ec8e666e-3afd-424b-b9ac-f8611854c168',
 4: '8a63530c-b4ec-4c33-a7f6-4cad94594fd9',
 5: '39ef682f-48a9-4b30-a91d-c610e2c78fcf',
 6: '9ddc13a3-60b8-44e4-a0e7-fc66e232944f',
 7: '1279b794-c33c-4558-9e75-f2b7d486d10f',
 8: '3a38c887-b908-4944-8e04-e222186f1da3',
 9: '987d340c-4064-412f-a7aa-5cdd7879683e',
 10: '11553bad-761b-446b-ba7d-c5890e16b239'}

In [31]:
vector_store.get_by_ids(['39ef682f-48a9-4b30-a91d-c610e2c78fcf'])

[Document(id='39ef682f-48a9-4b30-a91d-c610e2c78fcf', metadata={}, page_content="parameters and the enormous amount of training data, the scale of computation involved in training a large language model is mind-boggling. To illustrate, imagine that you could perform one billion additions and multiplications every single second. How long do you think it would take for you to do all of the operations involved in training the largest language models? Do you think it would take a year? Maybe something like 10,000 years? The answer is actually much more than that. It's well over 100 million years. This is only part of the story, though. This whole process is called pre-training. The goal of auto-completing a random passage of text from the internet is very different from the goal of being a good AI assistant. To address this, chatbots undergo another type of training, just as important, called reinforcement learning with human feedback. Workers flag unhelpful or problematic predictions, and 

## Step 2 - Retrieval

In [32]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [33]:
retriever

VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7a27d5529310>, search_kwargs={'k': 4})

In [34]:
retriever.invoke('What is Large Language Models')

[Document(id='f50f0d7c-e7e6-4ee2-bce7-57e92486dda9', metadata={}, page_content="does is assign a probability to all possible next words. To build a chatbot, what you do is lay out some text that describes an interaction between a user and a hypothetical AI assistant, you add on whatever the user types in as the first part of the interaction, and then have the model repeatedly predict the next word that such a hypothetical AI assistant would say in response, and that's what's presented to the user. In doing this, the output tends to look a lot more natural if you allow it to select less likely words along the way at random. So what this means is even though the model itself is deterministic, a given prompt typically gives a different answer each time it's run. Models learn how to make these predictions by processing an enormous amount of text, typically pulled from the internet. For a standard human to read the amount of text that was used to train GPT-3, for example, if they read non-s

## Step 3 - Augmentation

In [36]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.2
)

In [37]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [44]:
question          = "is the topic of Large Language Models discussed in this video? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)

In [45]:
retrieved_docs

[Document(id='987d340c-4064-412f-a7aa-5cdd7879683e', metadata={}, page_content="probability for every possible next word. Although researchers design the framework for how each of these steps work, it's important to understand that the specific behavior is an emergent phenomenon based on how those hundreds of billions of parameters are tuned during training. This makes it incredibly challenging to determine why the model makes the exact predictions that it does. What you can see is that when you use large language model predictions to autocomplete a prompt, the words that it generates are uncannily fluent, fascinating, and even useful. If you're a new viewer and you're curious about more details on how transformers and attention work, boy do I have some material for you. One option is to jump into a series I made about deep learning, where we visualize and motivate the details of attention and all the other steps in a transformer. Also, on my second channel I just posted a talk I gave 

In [46]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"probability for every possible next word. Although researchers design the framework for how each of these steps work, it's important to understand that the specific behavior is an emergent phenomenon based on how those hundreds of billions of parameters are tuned during training. This makes it incredibly challenging to determine why the model makes the exact predictions that it does. What you can see is that when you use large language model predictions to autocomplete a prompt, the words that it generates are uncannily fluent, fascinating, and even useful. If you're a new viewer and you're curious about more details on how transformers and attention work, boy do I have some material for you. One option is to jump into a series I made about deep learning, where we visualize and motivate the details of attention and all the other steps in a transformer. Also, on my second channel I just posted a talk I gave a couple months ago about this topic for the company TNG in Munich. Sometimes I

In [47]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [48]:
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      probability for every possible next word. Although researchers design the framework for how each of these steps work, it's important to understand that the specific behavior is an emergent phenomenon based on how those hundreds of billions of parameters are tuned during training. This makes it incredibly challenging to determine why the model makes the exact predictions that it does. What you can see is that when you use large language model predictions to autocomplete a prompt, the words that it generates are uncannily fluent, fascinating, and even useful. If you're a new viewer and you're curious about more details on how transformers and attention work, boy do I have some material for you. One option is to jump into a series I made about deep learning, where we visualize and motivate the details

## Step 4 - Generation

In [49]:
answer = llm.invoke(final_prompt)
print(answer.content)

Yes, the topic of Large Language Models is discussed in the video.

Here's what was discussed:
*   **Behavior and Predictions:** Their predictions are described as uncannily fluent, fascinating, and useful. It's challenging to determine why they make exact predictions, as their specific behavior is an emergent phenomenon based on how billions of parameters are tuned during training.
*   **Parameters/Weights:** What makes them "large" is their hundreds of billions of parameters (or weights). These parameters determine the probabilities the model gives for the next word.
*   **Training Process:**
    *   No human deliberately sets these parameters; they begin at random and are repeatedly refined based on many example pieces of text.
    *   Training involves passing all but the last word of an example into the model and comparing its prediction with the true last word.
    *   An algorithm called backpropagation tweaks parameters to make the model more likely to choose the true last word

## Building a Chain

In [50]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [51]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [52]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [53]:
parallel_chain.invoke('what LLM')

{'context': "does is assign a probability to all possible next words. To build a chatbot, what you do is lay out some text that describes an interaction between a user and a hypothetical AI assistant, you add on whatever the user types in as the first part of the interaction, and then have the model repeatedly predict the next word that such a hypothetical AI assistant would say in response, and that's what's presented to the user. In doing this, the output tends to look a lot more natural if you allow it to select less likely words along the way at random. So what this means is even though the model itself is deterministic, a given prompt typically gives a different answer each time it's run. Models learn how to make these predictions by processing an enormous amount of text, typically pulled from the internet. For a standard human to read the amount of text that was used to train GPT-3, for example, if they read non-stop 24-7, it would take over 2600 years. Larger models since then t

In [54]:
parser = StrOutputParser()

In [55]:
main_chain = parallel_chain | prompt | llm | parser

In [56]:
main_chain.invoke('Can you summarize the video')

'This video segment discusses how large language models (LLMs) predict the next word by calculating probabilities, noting that their specific behavior is an emergent phenomenon from billions of tuned parameters, making it challenging to understand *why* they make certain predictions. It highlights that the words generated by LLMs are fluent, fascinating, and useful. For new viewers, resources are mentioned, including a deep learning series visualizing transformers and attention, and a talk posted on a second channel.\n\nThe segment also explains that transformers and most language models encode words as lists of numbers (continuous values) for training. Transformers are unique due to their "attention" operation, which allows these number lists to interact and refine word meanings based on context (e.g., distinguishing "bank" as a riverbank). Additionally, transformers include feed-forward neural networks to store language patterns learned during training.'